In [ ]:
import torch
from lieflow.groups import SEplus12

In [ ]:
seplus12 = SEplus12()

In [ ]:
def random_time_like(N, ε_stab=1e-4):
    vec = torch.randn(N, 3)
    vec[..., 0] = (1 + torch.randn(N)**2).sqrt().clamp_min(1 + ε_stab) * (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    # restrict such that |angle| < pi
    return vec * ((1 - 1000. * ε_stab) * torch.pi / inner_product(vec, vec, keepdim=True).sqrt()).clamp_max(1.)

def random_space_like(N):
    vec = torch.randn(N, 3)
    vec[..., 0] = torch.clamp_min(1/(1 + torch.randn(N)**2).sqrt(), 0.1) * (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def random_light_like(N):
    vec = torch.randn(N, 3)
    vec[..., 0] = (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def inner_product(vec1, vec2, keepdim=False):
    return (vec1 @ seplus12.soplus12.η * vec2).sum(-1, keepdim=keepdim)

In [ ]:
N = 2**14

In [ ]:
test_vec = random_time_like(N)
lightlike = inner_product(test_vec, test_vec) < 0.01
N_timelike = N - lightlike.sum()
translations = torch.randn(N_timelike, 3)
test_vec = test_vec[~lightlike]
lorentz = (test_vec[..., None, None] * seplus12.soplus12.lie_algebra_basis).sum(-3)
test_gen = torch.zeros(N_timelike, 4, 4)
seplus12.pack_translation_lorentz(test_gen, translations, lorentz);

In [ ]:
(seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().max()

In [ ]:
(seplus12.exp(seplus12.log(seplus12.exp(test_gen))) - seplus12.exp(test_gen)).abs().max()

In [ ]:
bad_set = (seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().amax(dim=(-2, -1)) > 0.01
(bad_set * 1.).mean()

In [ ]:
test_vec = random_space_like(N)
lightlike = inner_product(test_vec, test_vec) > -0.01
N_spacelike = N - lightlike.sum()
translations = torch.randn(N_spacelike, 3)
test_vec = test_vec[~lightlike]
lorentz = (test_vec[..., None, None] * seplus12.soplus12.lie_algebra_basis).sum(-3)
test_gen = torch.zeros(N_spacelike, 4, 4)
seplus12.pack_translation_lorentz(test_gen, translations, lorentz);

In [ ]:
(seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().max()

In [ ]:
(seplus12.exp(seplus12.log(seplus12.exp(test_gen))) - seplus12.exp(test_gen)).abs().max()

In [ ]:
bad_set = (seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().amax(dim=(-2, -1)) > 0.01
(bad_set * 1.).mean()